<a href="https://colab.research.google.com/github/ashfaq1192/tool_calling_using_gemini/blob/main/Intro_Tool_calling_7_Jan_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing langchain

In [1]:
!pip install langchain google-generativeai langchain_google_genai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 2.0 MB/s eta 0:00:00


Accessing API

In [2]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Calculator Tool defining

In [3]:
from langchain_core.tools import tool

@tool
def Calculator(a: int, b: int) -> int:
    """Multiply a and b."""
    print("function is called")
    return a * b

# News Tool Calling

In [4]:
!pip install newsapi-python==0.2.6

In [5]:
from newsapi import NewsApiClient

@tool
def get_news(query: str) -> str:
    """
    Gets the top 3 news articles based on a query using newsapi.org.
    """
    api_key = userdata.get('NEWSAPI_API_KEY')
    newsapi = NewsApiClient(api_key=api_key)
    try:
        top_headlines = newsapi.get_everything(q=query, language='en', page_size=3)
        articles = top_headlines.get('articles', [])
        if not articles:
            return "No news found for this query."
        news_text = "\n\n".join([f"Title: {article['title']}\nURL: {article['url']}\nDescription: {article.get('description', 'No description available')}" for article in articles])
        return news_text
    except Exception as e:
        return f"An error occurred: {e}"

# Wather Tool Calling

In [6]:
!pip install requests==2.32.3

In [7]:
import requests

@tool
def get_weather(city_name: str) -> str:
  """
  Gets the current weather for a given city using OpenWeatherMap.
  """
  api_key = userdata.get('OPENWEATHER_API_KEY')
  base_url = "http://api.openweathermap.org/data/2.5/weather?"
  units = "metric" # You can change this to "imperial" for Fahrenheit
  complete_url = f"{base_url}appid={api_key}&q={city_name}&units={units}"
  try:
    response = requests.get(complete_url)
    response.raise_for_status()  # Raise an exception for HTTP errors
    data = response.json()
    if data["cod"] != "404":
        main_data = data["main"]
        weather_description = data["weather"][0]["description"]
        temperature = main_data["temp"]
        humidity = main_data["humidity"]
        return f"The weather in {city_name} is: {weather_description}, Temperature: {temperature}°C, Humidity: {humidity}%"
    else:
               return "City not found."
  except requests.exceptions.RequestException as e:
       return f"An error occurred: {e}"

In [8]:
tools = [Calculator, get_news, get_weather]

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model = "gemini-2.0-flash-exp" , api_key=GOOGLE_API_KEY)

In [10]:
from langchain.agents import initialize_agent, AgentType

In [11]:
agent = initialize_agent(tools, llm , agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION )

<ipython-input-11-a14b04c8fa12>:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(tools, llm , agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION )


# Calculator

In [12]:
respone =  agent.invoke({"input":"what is 2 multiply by 3"})
respone["output"]

function is called


'2 multiplied by 3 is 6.'

# Getting Latest **News**

Getting Latest News:

In [13]:
respone =  agent.invoke({"input":"what are the latest news about Pakistan"})
print(respone["output"])

The latest news about Pakistan includes an attempted record-breaking firework show in Karachi.


# Open Weather Forcast

In [14]:
respone =  agent.invoke({"input":"What is the weather of Lahore"})
print(respone["output"])

The weather in Lahore is smoke with a temperature of 10.99°C and 71% humidity.
